# TickPredict: predict life stage and engorgement

Set `IMAGE_PATH` below, then run the cells in order. Each saved model returns its predicted class, a confidence score for that class, and scores for every supported class.

Life stage supports **adult female** and **nymph**. Engorgement supports **unengorged**, **semi-engorged**, and **fully engorged**. Confidence is the model's **uncalibrated softmax score**, not a measured probability that the answer is correct. Unsupported species or non-tick images still receive predictions. These Penn research models use the original bottom-4% crop and have not been independently validated on phone photos.

To train new models or inspect the saved validation results, open [train.ipynb](train.ipynb).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
WORK = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "scripts/common.py").is_file()
     and (folder / "configs/life_stage.json").is_file()),
    None,
)
if WORK is None:
    raise RuntimeError("Open this notebook with Jupyter started in the project directory.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
print("Project:", WORK)

## Choose an image and model set

Set an absolute image path or one relative to this project. Leave `IMAGE_PATH = None` to inspect the notebook without prediction.

`MODEL_SET = "current"` uses the locked development predictors in `configs/current_models.json`. `"baseline"` uses the original committed ResNet18 models. A path in `CHECKPOINTS` overrides that task's selection. Training never changes the selected models automatically.

Locked v1.0 weights live locally in `models/v1.0/` and are excluded from Git. If they are missing on another computer, restore the complete versioned model folder or choose `"baseline"`; the notebook will not silently substitute a different model. No source dataset or initialization download is needed for these saved-checkpoint predictions.


In [ ]:
import json
from common import digest

IMAGE_PATH = None  # Example: "photos/my_tick.jpg"
MODEL_SET = "current"  # Or "baseline"
CHECKPOINTS = {"life_stage": None, "engorgement_level": None}
results = None

if MODEL_SET not in {"current", "baseline"}:
    raise ValueError("MODEL_SET must be current or baseline.")
selection = json.loads((WORK / "configs/current_models.json").read_text())
print("Selected model set:", MODEL_SET)
if MODEL_SET == "current":
    print(selection["status"])
    for task, selected in selection["models"].items():
        print(task.replace("_", " ").capitalize() + ": " + selected["description"])

if IMAGE_PATH is None:
    print("Set IMAGE_PATH to your tick photo, then rerun this cell.")
else:
    image_path = Path(IMAGE_PATH).expanduser()
    if not image_path.is_absolute():
        image_path = WORK / image_path
    if not image_path.is_file():
        raise FileNotFoundError(f"Photo not found: {image_path}")

    import torch
    torch.set_num_threads(2)
    from inference import predict
    from evaluate_predictor import predict_image
    from predict_lifeStage import DEFAULT_CHECKPOINT as LIFE_BASELINE
    from predict_engorgementLevel import DEFAULT_CHECKPOINT as ENGORGEMENT_BASELINE

    baseline = {"life_stage": LIFE_BASELINE, "engorgement_level": ENGORGEMENT_BASELINE}
    results = {}
    for task in baseline:
        override = CHECKPOINTS[task]
        if override is not None:
            checkpoint = Path(override).expanduser()
            if not checkpoint.is_absolute():
                checkpoint = WORK / checkpoint
            results[task] = predict(image_path, checkpoint, expected_task=task)
        elif MODEL_SET == "baseline":
            results[task] = predict(image_path, baseline[task], expected_task=task)
        else:
            selected = selection["models"][task]
            for checkpoint, expected_hash in selected["checkpoint_sha256"].items():
                path = WORK / checkpoint
                if not path.is_file():
                    raise FileNotFoundError(f"Missing selected model: {path}. Restore its run folder or choose MODEL_SET = 'baseline'.")
                if digest(path) != expected_hash:
                    raise ValueError(f"Selected checkpoint fingerprint changed: {path}")
            results[task] = predict_image(image_path, selected["predictor"], device="cpu")

    for task, result in results.items():
        print(f"\n{task.replace('_', ' ').capitalize()}: {result['prediction']}")
        print(f"Confidence (uncalibrated): {result['confidence']:.1%}")
        for label, score in sorted(result["model_scores_uncalibrated"].items(), key=lambda item: -item[1]):
            print(f"  {label}: {score:.1%}")
    print("\nFull outputs, including model provenance, are available in results.")
